In [1]:
import base64
import io
import math
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency

# EDA
## Goal: Visualize data, identify anomalies, and pinpoint data patterns for tuning and transformation considerations.

In [ ]:
class EDA:

    def __init__(
        self,
        X_splits,
        y_splits,
        run_analysis=False,
        run_univariate=False,
        run_bivariate=False,
        run_multivariate=False,
    ):
        self.X_train = X_splits["X_train"]
        self.y_train = y_splits["y_train"]
        self.raw_train = pd.concat([self.X_train, self.y_train], axis=1)

        self.cat_features = self.raw_train.select_dtypes(include=["object", "category", "string"]).columns.drop("bin", errors="ignore")
        self.num_features = self.raw_train.select_dtypes(include=["number"]).columns

        if not any([
            run_analysis,
            run_univariate,
            run_bivariate,
            run_multivariate,
        ]):
            run_analysis = run_univariate = run_bivariate = run_multivariate = True

        if run_analysis:
            self._data_analysis()
        if run_univariate:
            self._univariable_visualization()
        if run_bivariate:
            self._bivariable_visualization()
        if run_multivariate:
            self._multivariable_visualization()

    def _data_analysis(self):
        head_train = self.raw_train.head()
        display(head_train)

        info_train = pd.DataFrame({
            "Data Type": self.raw_train.dtypes,
            "Non-Null Count": self.raw_train.notnull().sum(),
            "Null Count": self.raw_train.isnull().sum(),
            "Null %": (self.raw_train.isnull().mean() * 100).round(2),
            "Unique Values": self.raw_train.nunique()
        })
        display(info_train)

        shape_train = pd.DataFrame({
            "Count":[self.raw_train.shape[1], self.raw_train.shape[0], self.raw_train.duplicated().sum()]
        }, index=["Column Count", "Row Count", "Duplicate Row Count"],
        )
        display(shape_train)

        cat_summary = pd.DataFrame({col: pd.Series(self.raw_train[col].dropna().unique()) for col in self.cat_features})
        cat_summary = cat_summary.fillna("").style.hide()
        display(cat_summary)

        num_summary = self.raw_train[self.num_features].describe()
        num_summary.loc["q1 -- q2(median) -- q3"] = (
            num_summary.loc["25%"].round(2).astype(str)
            + " -- "
            + num_summary.loc["50%"].round(2).astype(str)
            + " -- "
            + num_summary.loc["75%"].round(2).astype(str)
        )
        num_summary.loc["max - min = range"] = (
            num_summary.loc["max"].round(2).astype(str)
            + " - "
            + num_summary.loc["min"].round(2).astype(str)
            + " = "
            + (num_summary.loc["max"] - num_summary.loc["min"]).round(2).astype(str)
        )
        num_summary.loc["scale (iqr)"] = (num_summary.loc["75%"] - num_summary.loc["25%"])
        num_summary.loc["skewness"] = self.raw_train[self.num_features].skew()
        num_summary = num_summary.loc[["mean", "std", "q1 -- q2(median) -- q3", "max - min = range", "scale (iqr)", "skewness"]].T
        display(num_summary)

        target_summary = pd.DataFrame({
            "Count": self.y_train.value_counts(),
            "%": (self.y_train.value_counts(normalize=True) * 100).round(2),
        })
        target_summary.index = ["No Stroke", "Stroke"]
        display(target_summary)

        # Filter strictly for continuous numerical features (excluding binary flags with <= 2 unique values)
        continuous_features = [col for col in self.num_features if self.raw_train[col].nunique() > 2]

        q1 = self.raw_train[continuous_features].quantile(0.25)
        q3 = self.raw_train[continuous_features].quantile(0.75)
        iqr = q3 - q1
        is_outlier = (self.raw_train[continuous_features] < (q1 - 1.5 * iqr)) | (
            self.raw_train[continuous_features] > (q3 + 1.5 * iqr)
        )
        outlier_summary = pd.DataFrame({"Outlier Count": is_outlier.sum(), "Outlier (%)": (is_outlier.mean() * 100).round(2),})
        display(outlier_summary)

    def _univariable_visualization(self):
        # UNIVARIATE ANALYSIS 1: NUMERICAL FEATURE DISTRIBUTIONS (HISTOGRAMS + KDE)
        n_cols = 3
        n_rows = math.ceil(len(self.num_features) / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows))
        axes = axes.flatten()
        for i, feat in enumerate(self.num_features):
            sns.histplot(data=self.raw_train, x=feat, kde=True, bins=30, ax=axes[i])
            axes[i].set_title(f"Distribution of {feat}", fontsize=11)
            axes[i].set_xlabel(feat)
            axes[i].set_ylabel("Count")
        for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
        plt.tight_layout()
        plt.show()

        # UNIVARIATE ANALYSIS 2: CATEGORICAL FEATURE FREQUENCY (COUNT PLOTS)
        n_rows = math.ceil(len(self.cat_features) / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows))
        axes = axes.flatten()
        for i, feat in enumerate(self.cat_features):
            sns.countplot(data=self.raw_train, x=feat, color="red", ax=axes[i], order=self.raw_train[feat].value_counts().index)
            axes[i].set_title(f"Value Counts of {feat}", fontsize=11)
            axes[i].set_xlabel(feat)
            axes[i].set_ylabel("Count")
        for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
        plt.tight_layout()
        plt.show()

    def _bivariable_visualization(self):
        # BIVARIATE ANALYSIS 1: PEARSON VS SPEARMAN CORRELATION COMPARISON TABLE
        pearson_corr = self.raw_train[self.num_features].corr(method="pearson")
        spearman_corr = self.raw_train[self.num_features].corr(method="spearman")
        association_comparison = pd.DataFrame({"Pearson": pearson_corr.unstack(), "Spearman": spearman_corr.unstack(), 
            "Abs_diff": (pearson_corr - spearman_corr).abs().unstack()})
        display(association_comparison)

        # BIVARIATE ANALYSIS 2: STROKE RATE BY BINNED NUMERICAL FEATURES (QUANTILE BINS)
        n_cols = 3
        n_rows = math.ceil(len(self.num_features) / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows))
        axes = axes.flatten()

        for i, feat in enumerate(self.num_features):
            binned_col = pd.qcut(self.raw_train[feat], q=10, duplicates="drop")
            rates = self.raw_train.groupby(binned_col, observed=False)[
                "stroke"
            ].mean()
            rates.index = rates.index.astype(str)
            rates.plot(kind="line", marker="o", ax=axes[i], color="navy")
            axes[i].set_title(f"Stroke Rate by {feat}", fontsize=11)
            axes[i].set_xlabel(feat)
            axes[i].set_ylabel("Stroke Rate")
            axes[i].tick_params(axis="x", rotation=45)
            axes[i].grid(True, linestyle="--", alpha=0.5)
        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])
        plt.tight_layout()
        plt.show()

    def _multivariable_visualization(self):
        # MULTIVARIATE ANALYSIS 1: CRAMÉR'S V ASSOCIATION HEATMAP (CATEGORICAL VS CATEGORICAL)
        cramers_matrix = pd.DataFrame([
            [1.0 if c1 == c2 else (lambda cm: (lambda n, r, k, phi2: 
                0.0 if min(k - 1 - (k - 1)**2 / (n - 1), r - 1 - (r - 1)**2 / (n - 1)) == 0 
                else np.sqrt(max(0, phi2 - (k - 1) * (r - 1) / (n - 1)) / min(k - 1 - (k - 1)**2 / (n - 1), r - 1 - (r - 1)**2 / (n - 1)))
            )(cm.sum().sum(), cm.shape[0], cm.shape[1], chi2_contingency(cm)[0] / cm.sum().sum()))(pd.crosstab(self.raw_train[c1], self.raw_train[c2]))
            for c2 in self.cat_features]
            for c1 in self.cat_features
        ], index=self.cat_features, columns=self.cat_features)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cramers_matrix, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1, linewidths=0.5, annot_kws={"size": 9}, cbar_kws={"shrink": 0.8, "label": "Cramér's V Association"})
        plt.title("EDA Heatmap: Cramér's V Association Matrix of Categorical Features", fontsize=13, pad=15)
        plt.xticks(rotation=45, ha="right")
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()

        # MULTIVARIATE ANALYSIS 2: CORRELATION MATRIX HEATMAP (NUMERICAL VS NUMERICAL)
        corr_matrix = self.raw_train[self.num_features].corr()
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5, annot_kws={"size": 8}, cbar_kws={"shrink": 0.8})
        plt.title(f"EDA Heatmap: Correlation Matrix of All {len(self.num_features)} Features", fontsize=14, pad=15)
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()

        # MULTIVARIATE ANALYSIS 3: CORRELATION RATIO ETA HEATMAP (NUMERICAL VS CATEGORICAL)
        num_cols = [col for col in self.num_features if col != "stroke"]
        num_cat_matrix = pd.DataFrame([
            [0.0 if np.var(self.raw_train[num_col]) == 0 else np.sqrt(max(0.0, 1.0 - (
                np.sum(
                    np.array([len(a) for a in [self.raw_train[num_col][pd.Categorical(self.raw_train[cat_col]).codes == i][~np.isnan(self.raw_train[num_col][pd.Categorical(self.raw_train[cat_col]).codes == i])] for i in range(len(pd.Categorical(self.raw_train[cat_col]).categories))] if len(a) > 0]) / len(self.raw_train[num_col]) *
                    np.array([np.var(a) for a in [self.raw_train[num_col][pd.Categorical(self.raw_train[cat_col]).codes == i][~np.isnan(self.raw_train[num_col][pd.Categorical(self.raw_train[cat_col]).codes == i])] for i in range(len(pd.Categorical(self.raw_train[cat_col]).categories))] if len(a) > 0])
                ) / np.var(self.raw_train[num_col])
            ))) 
            for cat_col in self.cat_features]
            for num_col in num_cols
        ], index=num_cols, columns=self.cat_features)
        plt.figure(figsize=(10, 8))
        sns.heatmap(num_cat_matrix, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1, linewidths=0.5, annot_kws={"size": 9}, cbar_kws={"shrink": 0.8, "label": r"Correlation Ratio ($\eta$)"})
        plt.title("EDA Heatmap: Association Matrix (Numerical vs Categorical)", fontsize=13, pad=15)
        plt.xlabel("Categorical Features", fontsize=11)
        plt.ylabel("Numerical Features", fontsize=11)
        plt.xticks(rotation=45, ha="right")
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()